In [4]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import time
from PIL import Image
from torchvision import transforms
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import precision_score, recall_score, f1_score
from src.data.dataset import FundusDataset

# 超参数配置
config = {
    'model_name': 'google/vit-base-patch16-224',
    'num_classes': 8,
    'class_names': ['正常', '糖尿病视网膜病变', '青光眼', '白内障',
                   '黄斑变性', '高血压视网膜病变', '近视', '其他'],
    'batch_size': 16,
    'epochs': 25,
    'learning_rate': 1e-5,
    'weight_decay': 0.05,
    'warmup_steps': 500,
    'gradient_accumulation_steps': 2,
    'max_grad_norm': 1.0,
    'save_dir': './checkpoints',
    'log_dir': './logs',
    'pretrained_path': r'C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224',
}

# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 数据路径
train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation images'
excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'

print("正在加载数据集...")

# 训练集
train_dataset = FundusDataset(train_dir, excel_dir, is_training=True)
# 验证集
val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
# 测试集
test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

# 创建DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    pin_memory=True if device.type == 'cuda' else False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=4,
    pin_memory=True if device.type == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=4,
    pin_memory=True if device.type == 'cuda' else False
)

# 加载预训练模型
local_model_path = config['pretrained_path']
processor = ViTImageProcessor.from_pretrained(local_model_path)

model = ViTForImageClassification.from_pretrained(
    local_model_path,
    num_labels=config['num_classes'],
    ignore_mismatched_sizes=True
)
model = model.to(device)

print("✅ 模型加载成功")

# 损失函数和优化器
pos_rates = [0.06, 0.08, 0.02, 0.04, 0.04, 0.10, 0.06, 0.72]
pos_weight = torch.tensor([1.0 / (r + 0.01) for r in pos_rates]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# 训练准备
best_val = 0
global_step = 0
best_accuracy = 0
best_val_f1 = 0

writer = SummaryWriter('logs')
train_start_time = time.time()

print("\n开始训练...")

# 训练循环
for epoch in range(config['epochs']):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    train_strict_correct = 0
    train_strict_total = 0

    train_process = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["epochs"]}')

    for image, labels, img_name in train_process:
        image = image.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(image)
        loss = criterion(outputs.logits, labels)

        probability = torch.sigmoid(outputs.logits)
        predict = (probability > 0.5).float()

        # 严格准确率（8个标签全对）
        strict_correct = (predict == labels).all(dim=1).sum().item()
        strict_total = labels.size(0)
        strict_acc = strict_correct / strict_total

        # 宽松准确率（每个标签独立计算）
        total_correct = (predict == labels).sum().item()
        total_elements = labels.numel()
        batch_acc = total_correct / total_elements

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_strict_correct += strict_correct
        train_strict_total += strict_total
        train_correct += total_correct
        train_total += total_elements

        train_process.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{batch_acc:.3f}',
            'strict': f'{strict_acc:.3f}'
        })

        writer.add_scalar('Batch/Train_Loss', loss.item(), global_step)
        writer.add_scalar('Batch/Train_Strict_Accuracy', strict_acc, global_step)
        writer.add_scalar('Batch/Train_Label_Accuracy', batch_acc, global_step)
        global_step += 1

    # 计算epoch平均值
    train_strict_accuracy = train_strict_correct / train_strict_total
    train_label_accuracy = train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)

    writer.add_scalar('Loss/Train', avg_train_loss, epoch)
    writer.add_scalar('Accuracy/Train_Strict', train_strict_accuracy, epoch)
    writer.add_scalar('Accuracy/Train_Label', train_label_accuracy, epoch)
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Hyperparameters/Learning_Rate', current_lr, epoch)

    print(f"\nEpoch {epoch + 1}:")
    print(f"  损失: {avg_train_loss:.4f}")
    print(f"  标签准确率: {train_label_accuracy:.3f}")
    print(f"  严格准确率: {train_strict_accuracy:.3f}")

    # 验证集F1计算
    model.eval()
    all_val_probs = []
    all_val_labels = []

    with torch.no_grad():
        for val_images, val_labels, _ in val_loader:
            val_images = val_images.to(device)
            val_outputs = model(val_images)
            val_probs = torch.sigmoid(val_outputs.logits).cpu().numpy()
            all_val_probs.append(val_probs)
            all_val_labels.append(val_labels.numpy())

    all_val_probs = np.vstack(all_val_probs)
    all_val_labels = np.vstack(all_val_labels)

    val_preds = (all_val_probs > 0.5).astype(int)
    val_f1 = f1_score(all_val_labels, val_preds, average='macro', zero_division=0)

    print(f"  验证集Macro F1: {val_f1:.4f}")

    # 保存最佳模型
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            'strict_accuracy': train_strict_accuracy,
            'label_accuracy': train_label_accuracy,
            'val_f1': val_f1,
            'config': config
        }, os.path.join(config['save_dir'], 'best_model_by_val_f1.pth'))
        print(f"  ✅ 保存验证集最佳模型！F1={val_f1:.4f}")

    # 保存最新模型
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_train_loss,
        'accuracy': train_strict_accuracy,
        'config': config
    }, os.path.join(config['save_dir'], 'latest_model.pth'))

    if train_strict_accuracy > best_accuracy:
        best_accuracy = train_strict_accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            'accuracy': train_strict_accuracy,
            'config': config
        }, os.path.join(config['save_dir'], 'best_model.pth'))

train_time = time.time() - train_start_time
print(f"\n🎉 训练完成！")
print(f"总训练时间: {train_time / 60:.2f} 分钟")
print(f"最佳验证集F1: {best_val_f1:.4f}")
writer.close()
print("✅ TensorBoard 日志已保存")

使用设备: cuda
正在加载数据集...


Some weights of ViTForImageClassification were not initialized from the model checkpoint at C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


训练集大小: 4906
验证集大小: 1046
测试集大小: 1048
✅ 模型加载成功

开始训练...


Epoch 1/25:   0%|          | 0/306 [00:00<?, ?it/s]


Epoch 1:
  损失: 1.1974
  标签准确率: 0.616
  严格准确率: 0.000
  验证集Macro F1: 0.3098
  ✅ 保存验证集最佳模型！F1=0.3098


...


Epoch 23:
  损失: 0.1489
  标签准确率: 0.965
  严格准确率: 0.765
  验证集Macro F1: 0.5025


Epoch 24/25:   0%|          | 0/306 [00:00<?, ?it/s]


Epoch 24:
  损失: 0.1487
  标签准确率: 0.964
  严格准确率: 0.757
  验证集Macro F1: 0.5035


Epoch 25/25:   0%|          | 0/306 [00:00<?, ?it/s]


Epoch 25:
  损失: 0.1249
  标签准确率: 0.972
  严格准确率: 0.807
  验证集Macro F1: 0.4920

🎉 训练完成！
总训练时间: 229.56 分钟
最佳验证集F1: 0.5131
✅ TensorBoard 日志已保存


In [6]:
"""
无需命令行的类别分布分析脚本
直接在 IPython/Jupyter 中运行
"""

import numpy as np

def check_class_distribution(y_train, y_val, class_names=None):
    """
    分析训练集和验证集的类别分布
    """
    # 判断标签格式
    if len(y_train.shape) > 1 and y_train.shape[1] > 1:
        # one-hot 编码
        train_dist = np.mean(y_train, axis=0)
        val_dist = np.mean(y_val, axis=0)
        train_counts = np.sum(y_train, axis=0)
        val_counts = np.sum(y_val, axis=0)
        n_classes = y_train.shape[1]
    else:
        # 整数编码
        n_classes = len(np.unique(np.concatenate([y_train, y_val])))
        train_dist = np.array([np.mean(y_train == i) for i in range(n_classes)])
        val_dist = np.array([np.mean(y_val == i) for i in range(n_classes)])
        train_counts = np.array([np.sum(y_train == i) for i in range(n_classes)])
        val_counts = np.array([np.sum(y_val == i) for i in range(n_classes)])
    
    if class_names is None:
        class_names = [f"类别{i}" for i in range(n_classes)]
    
    print("=" * 60)
    print("类别分布分析报告")
    print("=" * 60)
    
    print("\n📊 各类别在训练集中的比例:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {train_dist[i]:.3f} ({int(train_counts[i])} 样本)")
    
    print("\n📊 各类别在验证集中的比例:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {val_dist[i]:.3f} ({int(val_counts[i])} 样本)")
    
    # 计算分布差异
    diff = np.abs(train_dist - val_dist)
    print("\n📈 分布差异:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {diff[i]:.3f}")
    
    # 统计信息
    print("\n📉 统计摘要:")
    print("-" * 40)
    print(f"  训练集总样本数: {int(np.sum(train_counts))}")
    print(f"  验证集总样本数: {int(np.sum(val_counts))}")
    print(f"  类别数量: {n_classes}")
    print(f"  平均分布差异: {np.mean(diff):.3f}")
    
    # 检查不平衡
    imbalance_ratio = np.max(train_counts) / np.min(train_counts)
    print(f"\n⚠️  不平衡比率: {imbalance_ratio:.2f}")
    if imbalance_ratio > 2:
        print("  警告: 数据集存在明显不平衡！")
    
    return {
        'train_dist': train_dist,
        'val_dist': val_dist,
        'diff': diff,
        'train_counts': train_counts,
        'val_counts': val_counts,
        'imbalance_ratio': imbalance_ratio
    }

def generate_sample_data():
    """生成示例数据"""
    np.random.seed(42)
    n_classes = 5
    n_samples = 1000
    
    # 生成不平衡数据
    train_probs = np.array([0.4, 0.25, 0.15, 0.12, 0.08])
    y_train = np.random.choice(n_classes, size=n_samples, p=train_probs)
    y_val = np.random.choice(n_classes, size=n_samples//5, p=train_probs)
    
    # 转换为 one-hot
    y_train_onehot = np.eye(n_classes)[y_train]
    y_val_onehot = np.eye(n_classes)[y_val]
    
    return y_train_onehot, y_val_onehot, ['A', 'B', 'C', 'D', 'E']

# 在 IPython/Jupyter 中直接运行这部分
if __name__ == "__main__":
    print("运行示例数据...")
    y_train, y_val, class_names = generate_sample_data()
    check_class_distribution(y_train, y_val, class_names)

运行示例数据...
类别分布分析报告

📊 各类别在训练集中的比例:
----------------------------------------
  A: 0.421 (421 样本)
  B: 0.250 (250 样本)
  C: 0.130 (130 样本)
  D: 0.120 (120 样本)
  E: 0.079 (79 样本)

📊 各类别在验证集中的比例:
----------------------------------------
  A: 0.360 (72 样本)
  B: 0.205 (41 样本)
  C: 0.165 (33 样本)
  D: 0.150 (30 样本)
  E: 0.120 (24 样本)

📈 分布差异:
----------------------------------------
  A: 0.061
  B: 0.045
  C: 0.035
  D: 0.030
  E: 0.041

📉 统计摘要:
----------------------------------------
  训练集总样本数: 1000
  验证集总样本数: 200
  类别数量: 5
  平均分布差异: 0.042

⚠️  不平衡比率: 5.33
  警告: 数据集存在明显不平衡！
